<a href="https://colab.research.google.com/github/Velgarath/recomendador_juegos_bgg/blob/main/juegosBGG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import re
import requests
import pandas as pd
import xml.etree.ElementTree as ET
import time
import os
from google.colab import userdata
from google.colab import drive
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import numpy as np

In [ ]:
#PROCEDIMIENTO PARA MONTAR EL DRIVE DE GOOGLE

import os
import shutil
from google.colab import drive

# 1. Intentamos desmontar de forma limpia usando la herramienta de disco de Colab
try:
    drive.flush_and_unmount()
    print("Unidad desmontada limpiamente.")
except Exception as e:
    print("No se pudo desmontar de forma estándar (puede que no estuviera montado):", e)

# 2. Si quedan carpetas residuales en la ruta, las eliminamos a la fuerza
mountpoint = '/content/drive'
if os.path.exists(mountpoint):
    try:
        # Eliminamos la carpeta "drive" y todo su contenido conflictivo local
        shutil.rmtree(mountpoint)
        print(f"Carpeta residual '{mountpoint}' eliminada con éxito.")
    except Exception as e:
        print(f"No se pudo eliminar la carpeta {mountpoint} directamente: {e}")
        print("Intentando forzar la eliminación por consola...")
        # Alternativa por comandos de sistema si Python encuentra el directorio bloqueado
        !umount -l /content/drive
        !rm -rf /content/drive

# 3. Volvemos a montar desde cero de forma limpia
print("Volviendo a montar Drive...")
drive.mount(mountpoint)

Unidad desmontada limpiamente.
Volviendo a montar Drive...
Mounted at /content/drive


In [ ]:
#ESTE CODIGO ME RENUEVA LOS FICHEROS DE KAGGLE DE LA BGG Y ME LOS METE EN GOOGLE DRIVE
#Importacion de la variable de entorno para conectarme a KAGGLE y descargar los datos
try:
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_API_TOKEN')
    print("¡Tokens de Kaggle cargados de forma segura desde los secretos de Colab!")
except Exception as e:
    print("Error:",e)

#Descarga de los datos de la web de Kaggle
!kaggle datasets download -d threnjen/board-games-database-from-boardgamegeek -p "/content/drive/MyDrive/Colab Notebooks/IA Juegos/Datos/" --unzip


¡Tokens de Kaggle cargados de forma segura desde los secretos de Colab!
Dataset URL: https://www.kaggle.com/datasets/threnjen/board-games-database-from-boardgamegeek
License(s): CC-BY-SA-3.0
100% 148M/148M [00:00<00:00, 201MB/s]



In [ ]:
#LECTURA DE LOS DATOS CSV DE KAGGLE
#Leemos el fichero games.csv y lo metemos en un DataFrame
df_games = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/IA Juegos/Datos/games.csv")

In [ ]:
# LECTURA DE LAS COLECCIONES DE LOS USUARIOS DE LA BGG
# NOTA IMPORTANTE: Hemos importado los XML de la BGG con esta URL del navegador:
# https://boardgamegeek.com/xmlapi2/collection?username=velgarath&stats=1&version=1
# https://boardgamegeek.com/xmlapi2/collection?username=ximo_valencia&stats=1&version=1


#Esta funcion es necesaria para limpiar los ampersands del XML
#Es un problema conocido de la exportacion de los XML de la BGG
def limpia_ampersands(texto_xml):
    # Reemplaza & que NO sea parte de una entidad válida (&amp; &lt; &gt; &quot; &apos; &#123; &#x1F;)
    return re.sub(r'&(?!amp;|lt;|gt;|quot;|apos;|#\d+;|#x[0-9a-fA-F]+;)', '&amp;', texto_xml)

#drive.mount('/content/drive', force_remount=True)

#abrimos y leemos el fichero XML de la coleccion de Luis
f = open("/content/drive/MyDrive/Colab Notebooks/IA Juegos/Datos/collection_velgarath.xml","r")
coleccion_luis = limpia_ampersands(f.read())
f.close()

#abrimos y leemos el fichero XML de la coleccion de Ximo
f = open("/content/drive/MyDrive/Colab Notebooks/IA Juegos/Datos/collection_ximo.xml","r")
coleccion_ximo = limpia_ampersands(f.read())
f.close()



In [ ]:
# PARSEADO XML DE UNA COLECCION DE JUEGOS DE MESA IMPORTADO DE LA BGG

def convierte_xml_a_dataframe(respuesta):
    raiz = ET.fromstring(respuesta)
    lista_juegos = []

    for item in raiz.findall('item'):
        objectid = item.attrib.get('objectid')  # clave para el merge con Kaggle

        name = item.find('name').text if item.find('name') is not None else 'Desconocido'
        year = item.find('yearpublished').text if item.find('yearpublished') is not None else 'N/A'
        numplays = item.find('numplays').text if item.find('numplays') is not None else 'N/A'

        status = item.find('status')
        own = status.attrib.get('own') if status is not None else None
        wishlist = status.attrib.get('wishlist') if status is not None else None
        preordered = status.attrib.get('preordered') if status is not None else None

        # stats puede faltar según cómo se exportó el XML, así que protegemos todo
        stats = item.find('stats')
        if stats is not None:
            min_players = stats.attrib.get('minplayers')
            max_players = stats.attrib.get('maxplayers')
            playing_time = stats.attrib.get('playingtime')
            num_owned = stats.attrib.get('numowned')
            etiqueta_rating = stats.find('rating')
            user_rating = etiqueta_rating.attrib.get('value') if etiqueta_rating is not None else 'N/A'
            average_elem = stats.find('rating/average')
            avg_rating = average_elem.attrib.get('value') if average_elem is not None else None
        else:
            min_players = max_players = playing_time = num_owned = None
            user_rating = 'N/A'
            avg_rating = None

        lista_juegos.append({
            'BGGId': objectid,
            'Name': name,
            'YearPublished': year,
            'AvgRating': avg_rating,
            'UserRating': user_rating,
            'MinPlayers': min_players,
            'MaxPlayers': max_players,
            'PlayingTime': playing_time,
            'NumOwned': num_owned,
            'NumPlays': numplays,
            'Own': own,
            'Wishlist': wishlist,
            'Preordered': preordered,
        })

    df_lista_juegos = pd.DataFrame(lista_juegos)

    columns_to_convert = ['YearPublished', 'AvgRating', 'UserRating',
                          'MinPlayers', 'MaxPlayers',
                          'PlayingTime', 'NumOwned', 'NumPlays']
    df_lista_juegos[columns_to_convert] = df_lista_juegos[columns_to_convert].apply(pd.to_numeric, errors='coerce')
    df_lista_juegos['BGGId'] = pd.to_numeric(df_lista_juegos['BGGId'], errors='coerce').astype('Int64')

    return df_lista_juegos

# Convertimos el XML de Luis en un DataFrame formateado como queremos
df_luis = convierte_xml_a_dataframe(coleccion_luis)
df_ximo = convierte_xml_a_dataframe(coleccion_ximo)
#Hemos visto que el tipo BGG_ID es "Int64". Lo pasamos a "int64 para que sea igual de la BD de Kagle"
df_luis['BGGId'] = df_luis['BGGId'].astype('int64')
df_ximo['BGGId'] = df_ximo['BGGId'].astype('int64')

In [ ]:
#PREPARACION DEL DATA FRAME CON LOS DATOS DE ENTRENAMIENTO
#
# vamos a crear el conjunto de datos de entrenamiento
# Esta sentencia hace JOIN de los juegos SOLO PUNTUADOS de mi coleccion con la base de datos global. Asi tenemos los datos de entrenamiento en un solo DF
# Ademàs, se queda SOLO con los juegos de mi coleccion puntuados. Los no puntuados NO se incorporan a la lista.
#     Por eso es INNER Join, para indicar que no solo quiero filas en las que EXISTAN EN AMBAS LISTAS y sean coincidente en el valor BBGId. Left join valdria tambien.

# De mi  colección solo necesito la clave y mi nota de puntuación (el objetivo "y" de entrenamiento). Las identificamos con "cols_luis"
cols_luis = ['BGGId', 'UserRating']
df_train = df_luis.loc[df_luis['UserRating'].notna(), cols_luis].merge(
    df_games, on='BGGId', how='inner'
)

#Normalizamos unos datos de "edad recomendada" y  "language ease", para que los valores NA contengan la media del resto de datos.
df_train['ComAgeRec'] = df_train['ComAgeRec'].fillna(df_train['ComAgeRec'].median())
df_train['LanguageEase'] = df_train['LanguageEase'].fillna(df_train['LanguageEase'].median())

#YA TENEMOS DF_TRAIN

In [ ]:
#LANZAMOS EL MACHINE LEARNING - LINEAR REGRESSION
#Vamos a crear la matriz de parametros X y el vector Y de respuestas.
# El modelo será una regresion WX + Y que haremos con scikit_learn.

features = ['GameWeight', 'MinPlayers', 'MaxPlayers', 'MfgPlaytime',
            'ComMinPlaytime', 'ComMaxPlaytime', 'ComAgeRec', 'LanguageEase',
            'Cat:Thematic', 'Cat:Strategy', 'Cat:War', 'Cat:Family',
            'Cat:CGS', 'Cat:Abstract', 'Cat:Party', 'Cat:Childrens']

X = df_train[features]
y = df_train['UserRating']

#print(X.shape)   # X.shape es (170, 16) — 170 juegos, 16 features
#print(y.shape)   # Y.shape es (170,)  — 170 notas

#DATOS PREPARADOS.

#BENCHMARK DE MODELO TONTO
#Vamos a comparar los resultados con la "evaluacion tonta" de marcar todo como mi puntuacion media que es de 7.63 en la BGG
#Esto nos va a servir de benchmark, para ver cómo se comportan los modelos con este benchmark tonto.

# Baseline: predecir SIEMPRE la nota media, para todos los juegos
pred_baseline = np.full(shape=len(y), fill_value=y.mean())
# Con este modelo "tonto" de marcar todos los juegos como la media nos sale el siguiente MAE
mae_baseline = mean_absolute_error(y, pred_baseline)
# Con este modelo "tonto" de marcar todos los juegos como la media nos sale el siguiente MSE
mse_baseline = mean_squared_error(y, pred_baseline)
# Con este modelo "tonto" de marcar todos los juegos como la media nos sale el siguiente MAE
rmse_baseline = np.sqrt(mse_baseline)
print(f"Baseline (predecir la media):")
print(f"  MAE  = {mae_baseline:.3f}  (error medio, en puntos)")
print(f"  MSE  = {mse_baseline:.3f}  (en puntos², penaliza fallos grandes)")
print(f"  RMSE = {rmse_baseline:.3f}  (en puntos, pero castiga outliers)\n")

#AHORA EMPEZAMOS A INVOCAR MODELOS
modelo = LinearRegression()

# CV de 5 pliegues. scikit-learn usa MAE negativo por convención, así que negamos.
scores = cross_val_score(modelo, X, y, cv=5, scoring='neg_mean_absolute_error')
mae_modelo = -scores.mean()

print(f"MAE del modelo (validación cruzada): {mae_modelo:.3f}")
print(f"MAE del baseline:                    1.343")
print(f"MAE por pliegue: {(-scores).round(3)}")

modelo.fit(X, y)   # ahora sí, entrenamos con todo para inspeccionar los pesos
pesos = pd.Series(modelo.coef_, index=X.columns).sort_values()
print(pesos)

# Pipeline: escala (z-score) y LUEGO aplica Ridge. Todo dentro de cada pliegue del CV.
modelo_ridge = make_pipeline(StandardScaler(), Ridge(alpha=1.0))

scores_ridge = cross_val_score(modelo_ridge, X, y, cv=5, scoring='neg_mean_absolute_error')
mae_ridge = -scores_ridge.mean()

print(f"MAE Ridge (CV):      {mae_ridge:.3f}")
print(f"MAE LinearRegression: 1.204")
print(f"MAE baseline:         1.343")
print(f"Pliegues Ridge: {(-scores_ridge).round(3)}")
print(f"Pliegues Linear: [0.804 1.118 1.475 1.205 1.418]")

modelo_ridge.fit(X, y)
pesos_ridge = pd.Series(
    modelo_ridge.named_steps['ridge'].coef_, index=X.columns
).sort_values()
print(pesos_ridge)


Baseline (predecir la media):
  MAE  = 1.343  (error medio, en puntos)
  MSE  = 3.425  (en puntos², penaliza fallos grandes)
  RMSE = 1.851  (en puntos, pero castiga outliers)

MAE del modelo (validación cruzada): 1.204
MAE del baseline:                    1.343
MAE por pliegue: [0.804 1.118 1.475 1.205 1.418]
Cat:Childrens    -5.324737e+00
Cat:Thematic     -1.073977e+00
Cat:Family       -3.553728e-01
MaxPlayers       -1.238548e-01
ComAgeRec        -1.172760e-01
LanguageEase     -9.852341e-04
ComMinPlaytime   -8.790553e-04
MfgPlaytime      -4.476914e-05
ComMaxPlaytime   -4.476914e-05
Cat:CGS          -4.440892e-16
MinPlayers        7.960704e-02
Cat:Abstract      3.181757e-01
Cat:Strategy      5.592132e-01
Cat:Party         6.262467e-01
GameWeight        1.007791e+00
Cat:War           1.483576e+00
dtype: float64
MAE Ridge (CV):      1.200
MAE LinearRegression: 1.204
MAE baseline:         1.343
Pliegues Ridge: [0.805 1.109 1.471 1.204 1.41 ]
Pliegues Linear: [0.804 1.118 1.475 1.205 1.41

In [ ]:
#LINEAR REGRESSION - 2
#Vamos a hacer un ranking de los juegos que me podrian gustar más.
#Fijaremos más adelante el umbral. De momento hagamos un TOP 20
#Vamoa a preparar los siguientes datos
#X_train = la matriz de mis juegos puntuados, que son los datos de entrenamiento
#y_train = la puntuación dado a cada uno de los juegos que tengo puntuados.
#X_nuevos = la matriz de juegos de Kagle que NO tengo

features = ['GameWeight', 'MinPlayers', 'MaxPlayers', 'MfgPlaytime',
            'ComMinPlaytime', 'ComMaxPlaytime', 'ComAgeRec', 'LanguageEase',
            'Cat:Thematic', 'Cat:Strategy', 'Cat:War', 'Cat:Family',
            'Cat:CGS', 'Cat:Abstract', 'Cat:Party', 'Cat:Childrens']


# Todos los BGGId que ya tienes (puntuados o no) en tu colección
ids_luis = df_luis['BGGId']

# Candidatos: juegos de Kaggle que NO están en tu colección
df_recomendar = df_games[~df_games['BGGId'].isin(ids_luis)].copy()
print(f"Candidatos a recomendar: {len(df_recomendar)}")

X_train = df_train[features]        # entrada de ENTRENAMIENTO (tus 170 juegos)
y_train = df_train['UserRating']    # objetivo: tus notas

X_nuevos = df_recomendar[features]  # entrada de PREDICCIÓN (los 21.235 que no tienes)

# este vector saca las medianas de todos los parámetros X del set de en0trenamiento
# Lo hacemos para normalizar los NaN datos de juegos que no tengo "X_nuevos"
# Un tema importante es que normalizamos con datos DE ENTRENAMIENTO. Nunca con datos de prediccion.
medianas_train = X_train.median()
X_nuevos = X_nuevos.fillna(medianas_train)

#Ahora tenemos los datos preparados, que son los siguientes
#X_train = la matriz de mis juegos puntuados, que son los datos de entrenamiento
#y_train = la puntuación dado a cada uno de los juegos que tengo puntuados.
#X_nuevos = la matriz de juegos de Kagle que NO tengo

#Pasamos el modelo
# La validación cruzada y el modelo final, ahora con los nombres nuevos
modelo_ridge = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
scores_ridge = cross_val_score(modelo_ridge, X_train, y_train, cv=5,
                               scoring='neg_mean_absolute_error')
modelo_ridge.fit(X_train, y_train)

# Predecir la nota de cada juego candidato
df_recomendar['nota_predicha'] = modelo_ridge.predict(X_nuevos)

df_recomendar['posicion'] = df_recomendar['nota_predicha'] \
    .rank(ascending=False, method='min').astype(int)

MIN_VOTOS = 1000
#Generamos el ranking quitando los juegos con menos de MIN_VOTOS y ordenando de mayor a menor
# Filtramos SOLO al construir el ranking. df_recomendar queda intacto (los 21.235).
ranking = df_recomendar[df_recomendar['NumUserRatings'] >= MIN_VOTOS] \
              .sort_values('nota_predicha', ascending=False)



# Top 20 — mostramos las features que el modelo dijo que más te gustan,
# para que TÚ juzgues si las recomendaciones tienen sentido
columnas_a_ver = ["posicion", 'Name', 'nota_predicha', 'GameWeight', 'Cat:War', 'Cat:Strategy']
print(ranking[columnas_a_ver].head(20).to_string())



# Buscar juegos con "18" en el nombre dentro del ranking completo (sin filtrar por votos)
mask_18xx = df_recomendar['Name'].str.contains(r'^18\d\d', na=False, regex=True) \
            & (df_recomendar['Cat:War'] == 0)
cols = ["posicion", 'Name', 'nota_predicha', 'GameWeight', 'Cat:War', 'Cat:Strategy']
print(df_recomendar[mask_18xx].sort_values('nota_predicha', ascending=False)[cols].head(15).to_string())


Candidatos a recomendar: 21235
       posicion                                                               Name  nota_predicha  GameWeight  Cat:War  Cat:Strategy
209           9                                              Advanced Squad Leader      10.471072      4.7321        1             0
16460        12                                                  Imperial Struggle      10.372104      3.8931        1             1
18079        29                                                               Root      10.177603      3.7068        1             1
6733         39                              Advanced Squad Leader: Starter Kit #3      10.121157      3.9000        1             0
1187         42                                                 Star Fleet Battles      10.116326      4.1830        1             0
809          48                                                       Squad Leader      10.055717      4.0365        1             0
5944        144                       